In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01_build_fips_spine
# MAGIC **Bronze -> Silver.** Builds the canonical national county FIPS spine that every other source joins onto.
# MAGIC
# MAGIC **Input:** `canonical_national_county2020.txt` (Census 2020-vintage county reference file) in the bronze Volume.
# MAGIC **Output:** managed Delta table `silver_fips_spine`, one row per county or county-equivalent, keyed on `geoid`.
# MAGIC
# MAGIC ## Decisions (written down so they survive until week 3)
# MAGIC 1. **Join key is `geoid` (5-char state+county FIPS), never a name.**
# MAGIC 2. **Territories are kept, not deleted.** `is_state_or_dc` marks the 50 states + DC. Dropping rows would be silent shaping; a flag keeps the data and makes the scope choice visible.
# MAGIC 3. **`is_independent_city` = `CLASSFP = 'C7'` only.** In this file C7 covers the 38 VA cities, Baltimore, St. Louis, and Carson City (41 total). `H6` is a different thing: consolidated city-counties (Denver, San Francisco, Philadelphia, ...). They get their own flag.
# MAGIC 4. **Connecticut is NOT resolved here.** The file holds the 8 legacy counties (class H4) and none of the 9 planning regions (09110-09190). They are loaded as-is and tagged in `known_hazard`. The canonical-grain decision and crosswalk are a separate task before ACS ingestion.

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

CATALOG = "bootcamp_students"
SCHEMA = "dataexpert_student_viknim"

BRONZE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/bronze_landing/fips_reference/"
    "canonical_national_county2020.txt"
)
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_fips_spine"

EXPECTED_HEADER = "STATE|STATEFP|COUNTYFP|COUNTYNS|COUNTYNAME|CLASSFP|FUNCSTAT"
TERRITORY_CODES = ["PR", "GU", "VI", "AS", "MP", "UM"]
KNOWN_CLASSFP = ["C7", "H1", "H4", "H5", "H6"]

# Expected counts for the file as received (Oct 2026). If you re-pull the source
# from Census, re-check these against the new file before changing them.
EXPECTED_TOTAL_ROWS = 3235
EXPECTED_STATE_DC_ROWS = 3143   # 3,143 because CT is still 8 legacy counties (3,144 with the 9 planning regions)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Read bronze with an explicit all-string schema
# MAGIC FIPS codes look numeric. If Spark infers types, `01` becomes `1` and every GEOID under state 10 silently breaks. Declare every column as string.
# MAGIC Because the schema is positional, we also check the header line matches what we expect.

# COMMAND ----------

header_line = spark.read.text(BRONZE_PATH).first()[0]
assert header_line == EXPECTED_HEADER, f"Unexpected header: {header_line!r}"

bronze_schema = StructType([
    StructField(name, StringType(), True)
    for name in ["STATE", "STATEFP", "COUNTYFP", "COUNTYNS", "COUNTYNAME", "CLASSFP", "FUNCSTAT"]
])

raw_df = (
    spark.read
    .option("sep", "|")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(bronze_schema)
    .csv(BRONZE_PATH)
)

raw_df.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Bronze sanity checks (stop here if any fail)

# COMMAND ----------

raw_count = raw_df.count()
bad_lengths = raw_df.filter(
    (F.length("STATEFP") != 2) | (F.length("COUNTYFP") != 3)
).count()

print(f"rows: {raw_count}")
print(f"rows with STATEFP != 2 chars or COUNTYFP != 3 chars: {bad_lengths}")

assert raw_count == EXPECTED_TOTAL_ROWS, f"Expected {EXPECTED_TOTAL_ROWS} rows, got {raw_count}"
assert all(isinstance(f.dataType, StringType) for f in raw_df.schema.fields), "A column is not string"
assert bad_lengths == 0, "FIPS parts have wrong lengths (leading zeros lost?)"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Build the silver spine
# MAGIC - `state_fips` (2 chars), `county_fips` (3 chars), `geoid` (5 chars = state + county)
# MAGIC - entity-type flags derived from `CLASSFP`; `FUNCSTAT` kept as a second independent signal
# MAGIC - `known_hazard` is populated where we already know a join will misbehave

# COMMAND ----------

CT_HAZARD = (
    "Legacy county (class H4). ACS 2023+ reports Connecticut as 9 planning regions "
    "(09110-09190), which are not in this file; Zillow still uses legacy counties. "
    "Raw FIPS join will silently miss. Crosswalk required."
)

spine_df = (
    raw_df
    .select(
        F.col("STATE").alias("state_abbr"),
        F.col("STATEFP").alias("state_fips"),
        F.col("COUNTYFP").alias("county_fips"),
        F.concat(F.col("STATEFP"), F.col("COUNTYFP")).alias("geoid"),
        F.col("COUNTYNS").alias("countyns"),
        F.col("COUNTYNAME").alias("county_name"),
        F.col("CLASSFP").alias("class_fp"),
        F.col("FUNCSTAT").alias("func_stat"),
    )
    .withColumn("is_independent_city", F.col("class_fp") == F.lit("C7"))
    .withColumn("is_consolidated_city_county", F.col("class_fp") == F.lit("H6"))
    .withColumn("is_census_area", F.col("class_fp") == F.lit("H5"))
    .withColumn("is_inactive_county", F.col("class_fp") == F.lit("H4"))
    .withColumn("is_state_or_dc", ~F.col("state_abbr").isin(TERRITORY_CODES))
    .withColumn(
        "known_hazard",
        F.when(F.col("state_abbr") == F.lit("CT"), F.lit(CT_HAZARD)),
    )
    .withColumn("_source_file", F.lit(BRONZE_PATH))
    .withColumn("_ingested_at", F.current_timestamp())
)

display(spine_df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Write the managed Delta table

# COMMAND ----------

(
    spine_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

spark.sql(
    f"COMMENT ON TABLE {SILVER_TABLE} IS "
    "'Canonical county/county-equivalent FIPS spine. Source: Census 2020-vintage "
    "national_county2020.txt. Join on geoid, never on name. Territories kept; filter with is_state_or_dc. "
    "Connecticut is legacy counties only (see known_hazard).'"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Constraints that Unity Catalog actually enforces
# MAGIC `NOT NULL` and `CHECK` are enforced at write time. `PRIMARY KEY` / `FOREIGN KEY` are informational only and would **not** stop a duplicate GEOID, so uniqueness is tested explicitly in section 6.

# COMMAND ----------

for col_name in ["state_abbr", "state_fips", "county_fips", "geoid", "county_name", "class_fp"]:
    spark.sql(f"ALTER TABLE {SILVER_TABLE} ALTER COLUMN {col_name} SET NOT NULL")

# DROP IF EXISTS first so this cell can be re-run.
spark.sql(f"ALTER TABLE {SILVER_TABLE} DROP CONSTRAINT IF EXISTS geoid_len_5")
spark.sql(f"ALTER TABLE {SILVER_TABLE} ADD CONSTRAINT geoid_len_5 CHECK (length(geoid) = 5)")

spark.sql(f"ALTER TABLE {SILVER_TABLE} DROP CONSTRAINT IF EXISTS fips_part_lengths")
spark.sql(
    f"ALTER TABLE {SILVER_TABLE} ADD CONSTRAINT fips_part_lengths "
    "CHECK (length(state_fips) = 2 AND length(county_fips) = 3)"
)

display(spark.sql(f"SHOW TBLPROPERTIES {SILVER_TABLE}"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Repeatable validation
# MAGIC Re-run this cell after every rebuild. It collects every result and fails at the end, so you see all problems at once instead of one at a time.

# COMMAND ----------

silver = spark.table(SILVER_TABLE)
rows = silver.select("geoid", "state_abbr", "class_fp").collect()
cls = {r.geoid: r.class_fp for r in rows}   # geoid -> class_fp (3,235 rows, tiny)

results = []

def check(name, passed, detail=""):
    results.append((name, bool(passed), str(detail)))

# --- structure -------------------------------------------------------------
total = silver.count()
distinct_geoids = silver.select("geoid").distinct().count()
dupes = silver.groupBy("geoid").count().filter("count > 1").count()   # PK is not enforced, so test it
state_dc_rows = silver.filter("is_state_or_dc").count()
null_keys = silver.filter(
    F.col("geoid").isNull() | F.col("state_fips").isNull() | F.col("county_fips").isNull()
    | F.col("county_name").isNull() | F.col("class_fp").isNull()
).count()

check("total rows", total == EXPECTED_TOTAL_ROWS, f"{total} (expected {EXPECTED_TOTAL_ROWS})")
check("50 states + DC rows", state_dc_rows == EXPECTED_STATE_DC_ROWS, f"{state_dc_rows} (expected {EXPECTED_STATE_DC_ROWS})")
check("no duplicate geoid", dupes == 0 and distinct_geoids == total, f"dupe groups={dupes}, distinct={distinct_geoids}")
check("no nulls in key columns", null_keys == 0, null_keys)
check("class_fp values all known", set(cls.values()) <= set(KNOWN_CLASSFP), sorted(set(cls.values())))

# --- Virginia --------------------------------------------------------------
va_class = {r["class_fp"]: r["count"] for r in
            silver.filter("state_abbr = 'VA'").groupBy("class_fp").count().collect()}
check("VA total = 133", sum(va_class.values()) == 133, va_class)
check("VA = 95 counties (H1) + 38 independent cities (C7)",
      va_class.get("H1") == 95 and va_class.get("C7") == 38, va_class)
check("VA Bedford city (51515) absent (retired 2013)", "51515" not in cls)

# --- independent cities nationally ----------------------------------------
n_ind = silver.filter("is_independent_city").count()
check("41 independent cities (38 VA + Baltimore + St. Louis + Carson City)", n_ind == 41, n_ind)
check("Carson City (32510) flagged independent", cls.get("32510") == "C7", cls.get("32510"))

# --- name-twin pairs: both rows exist, distinct geoids, correct types ------
TWINS = {
    "Fairfax, VA":   ("51059", "51600"),
    "Franklin, VA":  ("51067", "51620"),
    "Richmond, VA":  ("51159", "51760"),
    "Roanoke, VA":   ("51161", "51770"),
    "Baltimore, MD": ("24005", "24510"),
    "St. Louis, MO": ("29189", "29510"),
}
for label, (county_id, city_id) in TWINS.items():
    ok = cls.get(county_id) == "H1" and cls.get(city_id) == "C7" and county_id != city_id
    check(f"twin pair {label}: county {county_id} (H1) / city {city_id} (C7)", ok,
          f"{cls.get(county_id)} / {cls.get(city_id)}")

# --- Alaska ----------------------------------------------------------------
check("AK Valdez-Cordova (02261) absent", "02261" not in cls)
check("AK Chugach (02063) and Copper River (02066) present", "02063" in cls and "02066" in cls)

# --- Connecticut tripwire --------------------------------------------------
# Intentionally asserts the CURRENT state: 8 legacy counties, no planning regions.
# When you make the CT decision and add planning regions, update this check on purpose.
ct_rows = [g for g, _ in cls.items() if g.startswith("09")]
ct_planning = [g for g in ct_rows if "09110" <= g <= "09190"]
check("CT = 8 legacy counties, 0 planning regions (known hazard)", len(ct_rows) == 8 and len(ct_planning) == 0,
      f"{len(ct_rows)} CT rows, {len(ct_planning)} planning-region codes")

# --- report ----------------------------------------------------------------
report = spark.createDataFrame(results, ["check", "passed", "detail"])
display(report)

failed = [name for name, passed, _ in results if not passed]
assert not failed, f"{len(failed)} check(s) failed: {failed}"
print(f"All {len(results)} checks passed.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Next
# MAGIC - Decide Connecticut's canonical grain (planning regions vs. legacy counties) and land the Census crosswalk as a second bronze file. Update the CT tripwire above when you do.
# MAGIC - Then week 2: ACS and ZORI reshape, left-joined onto this spine. Always left-join from the spine, never inner-join, so coverage gaps stay visible.